In [2]:
"""
Download zonal weather for the 9 representative coordinates.

Part 1: observed weather (ERA5 archive API), 2013-01-01 -> 2026-05-31,
        chunked by year, cached per zone-year.
Part 2: lead-matched forecast weather (Previous Runs API), previous_day1..5,
        2024-01-23 -> 2026-05-31, chunked by month, cached per zone-month.

Outputs:
  weather_observed_zonal.parquet   columns like  temperature_2m__A_WEST
  weather_leads_zonal.parquet      columns like  temperature_2m_prev_day3__A_WEST

Idempotent: cached chunks are skipped on re-run.
"""

import time
from pathlib import Path

import pandas as pd
import requests

# ----------------------------- configuration -----------------------------
COORDS = {
    "A_WEST":   (42.886, -78.878),   # Buffalo
    "B_GENESE": (43.157, -77.616),   # Rochester
    "C_CENTRL": (43.048, -76.147),   # Syracuse
    "D_NORTH":  (44.928, -74.892),   # Massena
    "E_MHKVL":  (43.101, -75.233),   # Utica
    "F_CAPITL": (42.653, -73.757),   # Albany
    "G_HUDVL":  (41.706, -73.921),   # Poughkeepsie
    "J_NYC":    (40.714, -74.006),   # Manhattan
    "K_LONGIL": (40.730, -73.210),   # Islip
}

VARIABLES = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "apparent_temperature", "precipitation", "snowfall", "cloud_cover",
    "surface_pressure", "wind_speed_10m", "wind_gusts_10m",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
]

OBS_START, OBS_END = "2013-01-01", "2026-05-31"
LEAD_START, LEAD_END = "2024-01-23", "2026-05-31"
LEADS = [1, 2, 3, 4, 5]

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
PREVRUNS_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

CACHE = Path("weather_cache")
OBS_OUT = Path("weather_observed_zonal.parquet")
LEADS_OUT = Path("weather_leads_zonal.parquet")

SLEEP = 3.0          # seconds between API calls, polite to free tier


# ------------------------------ helpers ----------------------------------
def fetch(url: str, params: dict, retries: int = 3) -> dict:
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:            # rate limited: back off harder
                wait = 30 * (attempt + 1)
                print(f"    rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            r.raise_for_status()
        except requests.RequestException as e:
            print(f"    attempt {attempt + 1} failed ({e}), retrying...")
            time.sleep(10)
    raise RuntimeError(f"Failed after {retries} attempts: {params}")


def json_to_frame(js: dict, zone: str, rename_map: dict) -> pd.DataFrame:
    hourly = js["hourly"]
    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time")
    df.columns = [f"{rename_map.get(c, c)}__{zone}" for c in df.columns]
    return df


# ------------------------ part 1: observed weather ------------------------
def download_observed() -> pd.DataFrame:
    years = range(int(OBS_START[:4]), int(OBS_END[:4]) + 1)
    zone_frames = []
    for zone, (lat, lon) in COORDS.items():
        year_frames = []
        for yr in years:
            start = f"{yr}-01-01" if str(yr) != OBS_START[:4] else OBS_START
            end = f"{yr}-12-31" if str(yr) != OBS_END[:4] else OBS_END
            cache_file = CACHE / f"obs_{zone}_{yr}.parquet"
            if cache_file.exists():
                year_frames.append(pd.read_parquet(cache_file))
                continue
            js = fetch(ARCHIVE_URL, {
                "latitude": lat, "longitude": lon,
                "start_date": start, "end_date": end,
                "hourly": ",".join(VARIABLES),
                "timezone": "UTC",
            })
            df = json_to_frame(js, zone, {})
            df.to_parquet(cache_file)
            year_frames.append(df)
            time.sleep(SLEEP)
        zone_df = pd.concat(year_frames).sort_index()
        zone_df = zone_df[~zone_df.index.duplicated(keep="first")]
        zone_frames.append(zone_df)
        print(f"  observed done: {zone} ({len(zone_df):,} hours)")
    return pd.concat(zone_frames, axis=1)


# ---------------------- part 2: lead-matched weather ----------------------
def download_leads() -> pd.DataFrame:
    lead_vars = [f"{v}_previous_day{k}" for v in VARIABLES for k in LEADS]
    rename = {f"{v}_previous_day{k}": f"{v}_prev_day{k}"
              for v in VARIABLES for k in LEADS}
    months = pd.period_range(LEAD_START[:7], LEAD_END[:7], freq="M")

    zone_frames = []
    for zone, (lat, lon) in COORDS.items():
        month_frames = []
        for p in months:
            start = max(str(p.start_time.date()), LEAD_START)
            end = min(str(p.end_time.date()), LEAD_END)
            cache_file = CACHE / f"leads_{zone}_{p.strftime('%Y%m')}.parquet"
            if cache_file.exists():
                month_frames.append(pd.read_parquet(cache_file))
                continue
            js = fetch(PREVRUNS_URL, {
                "latitude": lat, "longitude": lon,
                "start_date": start, "end_date": end,
                "hourly": ",".join(lead_vars),
                "timezone": "UTC",
            })
            df = json_to_frame(js, zone, rename)
            df.to_parquet(cache_file)
            month_frames.append(df)
            time.sleep(SLEEP)
        zone_df = pd.concat(month_frames).sort_index()
        zone_df = zone_df[~zone_df.index.duplicated(keep="first")]
        zone_frames.append(zone_df)
        print(f"  leads done: {zone} ({len(zone_df):,} hours)")
    return pd.concat(zone_frames, axis=1)


# --------------------------------- main -----------------------------------
def main() -> None:
    CACHE.mkdir(exist_ok=True)

    print("Part 1: observed weather (archive API)...")
    obs = download_observed()
    obs.to_parquet(OBS_OUT)
    print(f"Saved {OBS_OUT}: {obs.shape[0]:,} rows x {obs.shape[1]} cols "
          f"({OBS_OUT.stat().st_size / 1e6:.1f} MB)")

    print("\nPart 2: lead-matched forecasts (Previous Runs API)...")
    leads = download_leads()
    leads.to_parquet(LEADS_OUT)
    print(f"Saved {LEADS_OUT}: {leads.shape[0]:,} rows x {leads.shape[1]} cols "
          f"({LEADS_OUT.stat().st_size / 1e6:.1f} MB)")

    # ------------------------- diagnostics -------------------------
    print("\n================ DIAGNOSTICS ================")
    for name, df in [("observed", obs), ("leads", leads)]:
        nan_pct = 100 * df.isna().sum().sum() / df.size
        print(f"{name}: {df.index.min()} -> {df.index.max()}, "
              f"NaN cells: {nan_pct:.2f}%")
    print("=============================================")


if __name__ == "__main__":
    main()

Part 1: observed weather (archive API)...
  observed done: A_WEST (117,576 hours)


  observed done: B_GENESE (117,576 hours)
  observed done: C_CENTRL (117,576 hours)


  observed done: D_NORTH (117,576 hours)
  observed done: E_MHKVL (117,576 hours)


  observed done: F_CAPITL (117,576 hours)
  observed done: G_HUDVL (117,576 hours)


  observed done: J_NYC (117,576 hours)
  observed done: K_LONGIL (117,576 hours)


...

  leads done: F_CAPITL (20,640 hours)


  leads done: G_HUDVL (20,640 hours)


  leads done: J_NYC (20,640 hours)


  leads done: K_LONGIL (20,640 hours)


Saved weather_leads_zonal.parquet: 20,640 rows x 585 cols (11.9 MB)

================ DIAGNOSTICS ================
observed: 2013-01-01 00:00:00 -> 2026-05-31 23:00:00, NaN cells: 0.00%
leads: 2024-01-23 00:00:00 -> 2026-05-31 23:00:00, NaN cells: 0.01%
